# IEEE-CIS - SET 2

Replicates the source paper's pipeline: SMOTE applied before the train/test split on a 50/50 balanced pool. Kept separate from SET 1 because this test set is SMOTE-contaminated by design (used for paper comparison, not for the leakage-free result).

Requires `preprocessed.pkl` from `preprocessing.ipynb`.

In [ ]:
# load preprocessed data
import pandas as pd
import numpy as np
import pickle

with open("preprocessed.pkl", "rb") as f:
    data = pickle.load(f)

df_clean = data["df_clean"]
df_train = data["df_train"]
df_test = data["df_test"]
X_train_smote = data["X_train_smote"]
y_train_smote = data["y_train_smote"]
TARGET_FRAUD = data["TARGET_FRAUD"]
TARGET_LEGIT = data["TARGET_LEGIT"]

print("Preprocessed data loaded.")

In [ ]:
# paper replication: SMOTE before split, 50/50 balanced pool (96K)
from imblearn.over_sampling import SMOTENC
from sklearn import config_context

TOTAL_POOL_PAPER = 96000
TARGET_FRAUD_PAPER = 48000   
TARGET_LEGIT_PAPER = 48000

fraud_pool_all = df_clean[df_clean['isFraud'] == 1]
legit_pool_all = df_clean[df_clean['isFraud'] == 0]

print("Real fraud available:", len(fraud_pool_all))
print("Real legit available:", len(legit_pool_all))

FRAUD_BASE_CAP = 10000
fraud_sample_paper = fraud_pool_all.sample(
    n=min(FRAUD_BASE_CAP, len(fraud_pool_all)), random_state=42
)
legit_sample_paper = legit_pool_all.sample(n=TARGET_LEGIT_PAPER, random_state=42)

df_pool_paper = pd.concat([fraud_sample_paper, legit_sample_paper])
df_pool_paper = df_pool_paper.sample(frac=1, random_state=42)

print("\nPre-SMOTE pool (paper-style, 50% target):")
print("Fraud (real, capped):", len(fraud_sample_paper))
print("Legit (real):", len(legit_sample_paper))
print("Synthetic fraud needed:", TARGET_FRAUD_PAPER - len(fraud_sample_paper))

X_pool_paper = df_pool_paper.drop(columns=['isFraud'])
y_pool_paper = df_pool_paper['isFraud']

num_cols_paper = X_pool_paper.select_dtypes(include=[np.number]).columns
X_pool_paper[num_cols_paper] = X_pool_paper[num_cols_paper].astype(np.float32)

cat_cols_paper = X_pool_paper.select_dtypes(include=['object']).columns.tolist()
cat_idx_paper = [X_pool_paper.columns.get_loc(c) for c in cat_cols_paper]

smote_paper = SMOTENC(
    categorical_features=cat_idx_paper,
    sampling_strategy=TARGET_FRAUD_PAPER / TARGET_LEGIT_PAPER,
    random_state=42,
    k_neighbors=5
)

with config_context(working_memory=64):
    X_smote_paper, y_smote_paper = smote_paper.fit_resample(X_pool_paper, y_pool_paper)

print("\nAfter SMOTE (full balanced pool):")
print("Legit:", (y_smote_paper==0).sum(), "| Fraud:", (y_smote_paper==1).sum())
print("Total:", len(y_smote_paper))

In [ ]:
# one-hot encode the balanced pool
cat_cols_paper2 = X_smote_paper.select_dtypes(include=['object']).columns.tolist()

X_smote_encoded_paper = pd.get_dummies(
    X_smote_paper, columns=cat_cols_paper2, dtype=float
)

print("Encoded shape:", X_smote_encoded_paper.shape)

In [ ]:
# split into train/val/test (57K/15K/24K)
from sklearn.model_selection import train_test_split

X_temp_paper, X_test_paper, y_temp_paper, y_test_paper = train_test_split(
    X_smote_encoded_paper, y_smote_paper,
    test_size=24000/96000,
    random_state=42,
    stratify=y_smote_paper
)

X_train_paper, X_val_paper, y_train_paper, y_val_paper = train_test_split(
    X_temp_paper, y_temp_paper,
    test_size=15000/72000,
    random_state=42,
    stratify=y_temp_paper
)

X_train_paper = X_train_paper.values
X_val_paper   = X_val_paper.values
X_test_paper  = X_test_paper.values
y_train_paper = np.array(y_train_paper)
y_val_paper   = np.array(y_val_paper)
y_test_paper  = np.array(y_test_paper)

print("Train:", X_train_paper.shape, "| Fraud %:", round(y_train_paper.mean()*100,1))
print("Val:  ", X_val_paper.shape,   "| Fraud %:", round(y_val_paper.mean()*100,1))
print("Test: ", X_test_paper.shape,  "| Fraud %:", round(y_test_paper.mean()*100,1))

In [ ]:
# logistic regression baseline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score,
                              confusion_matrix)

print("Training Logistic Regression (paper pipeline)...")

lr_model_paper = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
lr_model_paper.fit(X_train_paper, y_train_paper)

lr_pred_val_paper = lr_model_paper.predict(X_val_paper)
lr_prob_val_paper = lr_model_paper.predict_proba(X_val_paper)[:, 1]

print("Logistic Regression — Validation Results (paper pipeline):")
print("Accuracy: ", round(accuracy_score(y_val_paper, lr_pred_val_paper)*100, 2), "%")
print("Precision:", round(precision_score(y_val_paper, lr_pred_val_paper)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val_paper, lr_pred_val_paper)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val_paper, lr_pred_val_paper)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val_paper, lr_prob_val_paper), 4))
print(confusion_matrix(y_val_paper, lr_pred_val_paper))

In [ ]:
# build ResNeXt-GRU model
import tensorflow as tf
from tensorflow.keras import layers, models

def build_rxt_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)

    path1 = layers.Dense(32, activation='relu')(x)
    path1 = layers.BatchNormalization()(path1)
    path2 = layers.Dense(32, activation='relu')(x)
    path2 = layers.BatchNormalization()(path2)
    path3 = layers.Dense(32, activation='relu')(x)
    path3 = layers.BatchNormalization()(path3)
    path4 = layers.Dense(32, activation='relu')(x)
    path4 = layers.BatchNormalization()(path4)

    merged = layers.concatenate([path1, path2, path3, path4])
    projected = layers.Dense(input_dim, activation='linear')(merged)
    rxt_out = layers.Activation('relu')(layers.add([x, projected]))

    gru_out = layers.GRU(128, dropout=0.3, return_sequences=False)(rxt_out)

    dense = layers.Dense(64, activation='relu')(gru_out)
    dropout = layers.Dropout(0.3)(dense)
    output = layers.Dense(1, activation='sigmoid')(dropout)

    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.AUC(name='auc')]
    )
    return model

input_dim_paper = X_train_paper.shape[1]
rxt_model_paper = build_rxt_model(input_dim_paper)
rxt_model_paper.summary()

In [ ]:
# train ResNeXt-GRU model
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights_paper = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train_paper
)
cw_paper = {0: class_weights_paper[0], 1: class_weights_paper[1]}
print("Class weights:", cw_paper)

callbacks_rxt_paper = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001, verbose=1
    )
]

print("Training ResNeXt-GRU (paper pipeline)...")

history_rxt_paper = rxt_model_paper.fit(
    X_train_paper, y_train_paper,
    validation_data=(X_val_paper, y_val_paper),
    epochs=50,
    batch_size=32,
    class_weight=cw_paper,
    callbacks=callbacks_rxt_paper,
    verbose=1
)

print("ResNeXt-GRU training complete")
print("Stopped at epoch:", len(history_rxt_paper.history['loss']))

rxt_prob_val_paper = rxt_model_paper.predict(X_val_paper).flatten()
rxt_pred_val_paper = (rxt_prob_val_paper > 0.5).astype(int)

print("\nResNeXt-GRU — Validation Results (paper pipeline):")
print("Accuracy: ", round(accuracy_score(y_val_paper, rxt_pred_val_paper)*100, 2), "%")
print("Precision:", round(precision_score(y_val_paper, rxt_pred_val_paper)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val_paper, rxt_pred_val_paper)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val_paper, rxt_pred_val_paper)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val_paper, rxt_prob_val_paper), 4))
print(confusion_matrix(y_val_paper, rxt_pred_val_paper))

In [ ]:
# build ResNeXt-GRU + attention model
def build_rxt_attention_model(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Reshape((1, input_dim))(inputs)

    p1 = layers.Dense(64, activation='relu')(x)
    p1 = layers.BatchNormalization()(p1)
    p2 = layers.Dense(64, activation='relu')(x)
    p2 = layers.BatchNormalization()(p2)
    p3 = layers.Dense(64, activation='relu')(x)
    p3 = layers.BatchNormalization()(p3)
    p4 = layers.Dense(64, activation='relu')(x)
    p4 = layers.BatchNormalization()(p4)
    merged1 = layers.concatenate([p1, p2, p3, p4])
    proj1 = layers.Dense(input_dim, activation='linear')(merged1)
    block1_out = layers.Activation('relu')(layers.add([x, proj1]))

    p5 = layers.Dense(64, activation='relu')(block1_out)
    p5 = layers.BatchNormalization()(p5)
    p6 = layers.Dense(64, activation='relu')(block1_out)
    p6 = layers.BatchNormalization()(p6)
    p7 = layers.Dense(64, activation='relu')(block1_out)
    p7 = layers.BatchNormalization()(p7)
    p8 = layers.Dense(64, activation='relu')(block1_out)
    p8 = layers.BatchNormalization()(p8)
    merged2 = layers.concatenate([p5, p6, p7, p8])
    proj2 = layers.Dense(input_dim, activation='linear')(merged2)
    block2_out = layers.Activation('relu')(layers.add([block1_out, proj2]))

    gru_out = layers.GRU(128, dropout=0.3, return_sequences=True)(block2_out)

    att_out = layers.MultiHeadAttention(num_heads=4, key_dim=32)(gru_out, gru_out)
    att_norm = layers.LayerNormalization()(layers.add([gru_out, att_out]))

    flat = layers.Flatten()(att_norm)
    dense1 = layers.Dense(128, activation='relu')(flat)
    drop1 = layers.Dropout(0.3)(dense1)
    dense2 = layers.Dense(64, activation='relu')(drop1)
    drop2 = layers.Dropout(0.2)(dense2)
    output = layers.Dense(1, activation='sigmoid')(drop2)

    model = models.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall'),
                 tf.keras.metrics.AUC(name='auc')]
    )
    return model

rxt_att_model_paper = build_rxt_attention_model(input_dim_paper)
rxt_att_model_paper.summary()

In [ ]:
# train ResNeXt-GRU + attention model
callbacks_att_paper = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=0.00001, verbose=1
    )
]

print("Training ResNeXt-GRU + Attention (paper pipeline)...")

history_att_paper = rxt_att_model_paper.fit(
    X_train_paper, y_train_paper,
    validation_data=(X_val_paper, y_val_paper),
    epochs=50,
    batch_size=32,
    class_weight=cw_paper,
    callbacks=callbacks_att_paper,
    verbose=1
)

print("ResNeXt-GRU + Attention training complete")
print("Stopped at epoch:", len(history_att_paper.history['loss']))

att_prob_val_paper = rxt_att_model_paper.predict(X_val_paper).flatten()
att_pred_val_paper = (att_prob_val_paper > 0.5).astype(int)                                                     

print("\nResNeXt-GRU + Attention — Validation Results (paper pipeline):") 
print("Accuracy: ", round(accuracy_score(y_val_paper, att_pred_val_paper)*100, 2), "%")
print("Precision:", round(precision_score(y_val_paper, att_pred_val_paper)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val_paper, att_pred_val_paper)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val_paper, att_pred_val_paper)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val_paper, att_prob_val_paper), 4))
print(confusion_matrix(y_val_paper, att_pred_val_paper))

In [ ]:
# XGBoost baseline
from xgboost import XGBClassifier

print("Training XGBoost (paper pipeline)...")

xgb_model_paper = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=1.5
)
xgb_model_paper.fit(
    X_train_paper, y_train_paper,
    eval_set=[(X_val_paper, y_val_paper)],
    verbose=False
)

xgb_pred_val_paper = xgb_model_paper.predict(X_val_paper)
xgb_prob_val_paper = xgb_model_paper.predict_proba(X_val_paper)[:, 1]

print("XGBoost — Validation Results (paper pipeline):")
print("Accuracy: ", round(accuracy_score(y_val_paper, xgb_pred_val_paper)*100, 2), "%")
print("Precision:", round(precision_score(y_val_paper, xgb_pred_val_paper)*100, 2), "%")
print("Recall:   ", round(recall_score(y_val_paper, xgb_pred_val_paper)*100, 2), "%")
print("F1 Score: ", round(f1_score(y_val_paper, xgb_pred_val_paper)*100, 2), "%")
print("AUC-ROC:  ", round(roc_auc_score(y_val_paper, xgb_prob_val_paper), 4))
print(confusion_matrix(y_val_paper, xgb_pred_val_paper))

In [ ]:
# validation summary table (paper pipeline)
import pandas as pd

results_val_paper = {
    'Logistic Regression': [
        accuracy_score(y_val_paper, lr_pred_val_paper),
        precision_score(y_val_paper, lr_pred_val_paper, zero_division=0),
        recall_score(y_val_paper, lr_pred_val_paper, zero_division=0),
        f1_score(y_val_paper, lr_pred_val_paper, zero_division=0),
        roc_auc_score(y_val_paper, lr_prob_val_paper)
    ],

    'ResNeXt-GRU': [
        accuracy_score(y_val_paper, rxt_pred_val_paper),
        precision_score(y_val_paper, rxt_pred_val_paper, zero_division=0),
        recall_score(y_val_paper, rxt_pred_val_paper, zero_division=0),
        f1_score(y_val_paper, rxt_pred_val_paper, zero_division=0),
        roc_auc_score(y_val_paper, rxt_prob_val_paper)
    ],
    'ResNeXt-GRU + Attention': [
        accuracy_score(y_val_paper, att_pred_val_paper),
        precision_score(y_val_paper, att_pred_val_paper, zero_division=0),
        recall_score(y_val_paper, att_pred_val_paper, zero_division=0),
        f1_score(y_val_paper, att_pred_val_paper, zero_division=0),
        roc_auc_score(y_val_paper, att_prob_val_paper)
    ],
    'XGBoost': [
        accuracy_score(y_val_paper, xgb_pred_val_paper),
        precision_score(y_val_paper, xgb_pred_val_paper, zero_division=0),
        recall_score(y_val_paper, xgb_pred_val_paper, zero_division=0),
        f1_score(y_val_paper, xgb_pred_val_paper, zero_division=0),
        roc_auc_score(y_val_paper, xgb_prob_val_paper)
    ],
}

df_val_summary_paper = pd.DataFrame.from_dict(
    results_val_paper, orient='index',
    columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
)
print("=== VALIDATION SUMMARY — PAPER PIPELINE (50% fraud, SMOTE before split) ===")
display(df_val_summary_paper.round(4))

In [ ]:
# evaluate all models on the test set (SMOTE-contaminated by design)
results_test_paper = {}
preds_test_paper = {}
probs_test_paper = {}

lr_pred_test_paper = lr_model_paper.predict(X_test_paper)
lr_prob_test_paper = lr_model_paper.predict_proba(X_test_paper)[:, 1]
preds_test_paper['Logistic Regression'] = lr_pred_test_paper
probs_test_paper['Logistic Regression'] = lr_prob_test_paper

rxt_prob_test_paper = rxt_model_paper.predict(X_test_paper).flatten()
rxt_pred_test_paper = (rxt_prob_test_paper > 0.5).astype(int)
preds_test_paper['ResNeXt-GRU'] = rxt_pred_test_paper
probs_test_paper['ResNeXt-GRU'] = rxt_prob_test_paper

att_prob_test_paper = rxt_att_model_paper.predict(X_test_paper).flatten()
att_pred_test_paper = (att_prob_test_paper > 0.5).astype(int)
preds_test_paper['ResNeXt-GRU + Attention'] = att_pred_test_paper
probs_test_paper['ResNeXt-GRU + Attention'] = att_prob_test_paper

xgb_pred_test_paper = xgb_model_paper.predict(X_test_paper)
xgb_prob_test_paper = xgb_model_paper.predict_proba(X_test_paper)[:, 1]
preds_test_paper['XGBoost'] = xgb_pred_test_paper
probs_test_paper['XGBoost'] = xgb_prob_test_paper

for name in preds_test_paper:
    y_pred = preds_test_paper[name]
    y_prob = probs_test_paper[name]
    results_test_paper[name] = [
        accuracy_score(y_test_paper, y_pred),
        precision_score(y_test_paper, y_pred, zero_division=0),
        recall_score(y_test_paper, y_pred, zero_division=0),
        f1_score(y_test_paper, y_pred, zero_division=0),
        roc_auc_score(y_test_paper, y_prob)
    ]

df_test_summary_paper = pd.DataFrame.from_dict(
    results_test_paper, orient='index',
    columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
)
print("=== TEST SUMMARY — PAPER PIPELINE (50% fraud test set) ===")
display(df_test_summary_paper.round(4))

In [ ]:
# tune decision threshold per model
from sklearn.metrics import f1_score, precision_score, recall_score

def tune_threshold(name, y_true, y_prob, thresholds=[0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]):
    print(f"--- {name} ---")
    print(f"{'Threshold':<10} {'Recall':>8} {'Precision':>10} {'F1':>8} {'FalseAlarms':>12}")
    best_t, best_f1 = 0.5, 0
    for t in thresholds:
        preds = (y_prob > t).astype(int)
        rec  = recall_score(y_true, preds, zero_division=0)
        prec = precision_score(y_true, preds, zero_division=0)
        f1   = f1_score(y_true, preds, zero_division=0)
        fa   = ((preds == 1) & (y_true == 0)).sum()
        print(f"  {t:<8} {rec*100:>7.1f}% {prec*100:>9.1f}% {f1*100:>7.1f}% {fa:>11,}")
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f"Best threshold: {best_t} | Best F1: {best_f1*100:.2f}%\n")
    return best_t, best_f1

thresholds_summary_paper = {}
for name in probs_test_paper:
    best_t, best_f1 = tune_threshold(name, y_test_paper, probs_test_paper[name])
    thresholds_summary_paper[name] = {'best_threshold': best_t, 'best_f1': best_f1}

print("=== BEST THRESHOLDS SUMMARY (paper pipeline, 50% fraud) ===")
for name, vals in thresholds_summary_paper.items():
    print(f"{name:<28} threshold={vals['best_threshold']} F1={vals['best_f1']*100:.2f}%")

In [ ]:
# checkpoint: save V3 test results
import pickle
checkpoint_v3 = {
    'df_test_summary_paper': df_test_summary_paper,
}
with open("checkpoint_v3.pkl", "wb") as f:
    pickle.dump(checkpoint_v3, f)
print("V3 checkpoint saved.")

In [ ]:
# extract per-epoch training time from saved cell outputs
import json, re

def get_epoch_times(text):
    times = re.findall(r'\x1b\[1m(\d+)s\x1b\[0m', text)
    return [int(t) for t in times]

def get_output_text(cell):
    return ''.join(''.join(o.get('text', [])) for o in cell.get('outputs', []))

def find_model_cells(nb_path, keywords=('rxt_model_paper.fit', 'rxt_att_model_paper.fit')):
    nb = json.load(open(nb_path, encoding='utf-8'))
    print("="*20, nb_path, "="*20)
    for i, cell in enumerate(nb['cells']):
        src = ''.join(cell.get('source', []))
        for kw in keywords:
            if kw in src:
                print(i, '|', kw, '|', src.split(chr(10))[0][:70])
    print()

def extract_training_times(nb_path, model_cells, outlier_threshold=200):
    nb = json.load(open(nb_path, encoding='utf-8'))
    results = {}
    print("="*20, nb_path, "="*20)
    for idx, name in model_cells:
        text = get_output_text(nb['cells'][idx])
        times = get_epoch_times(text)
        raw_total = sum(times)
        clean_total = sum(t for t in times if t < outlier_threshold)
        n_outliers = sum(1 for t in times if t >= outlier_threshold)
        results[name] = {'epochs': len(times), 'raw_total_s': raw_total,
                          'clean_total_s': clean_total, 'outlier_epochs': n_outliers}
        print(f"{name:<28} epochs={len(times):>3} | raw={raw_total/60:>6.1f} min | "
              f"clean={clean_total/60:>6.1f} min | outliers={n_outliers}")
    print()
    return results

find_model_cells("IEEE-CIS-SET2.ipynb")

v3_times = extract_training_times("IEEE-CIS-SET2.ipynb", [
    (6, "ResNeXt-GRU"), (8, "ResNeXt-GRU + Attention")
])